In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('../../shared'))   # repo shared/ (paths.py); run this notebook from its folder
from paths import outdir as _outdir, neuropixels, NEUROPIXELS_CACHE
NP_DATA = os.path.join(neuropixels(), '')                       # was NP_DATA (Neuropixels intermediate data)
NP_SAVE = os.path.join(_outdir('fig4_motifs/legacy/save'), '')  # was NP_SAVE (CITS results)
from cits.methods import cits_full_weighted #RUN
import multiprocessing
import numpy as np
import time
import os
import pickle as pkl
import matplotlib.pyplot as plt

In [ ]:
# %% RUN
def smooth_spikes( P, kernel_width , bin_size ):
    # Rahul
    """
    P: #trials x #units x #time-points sized ndarray
    kernel_param: width of kernel (in samplingrate*1000 units)

    """

    import time
    import scipy.stats, scipy.signal
    import scipy as sp

    start = time.time() # tic Measures function running speed. Works only on Macs

    #trials = P.shape[0]
    #units = P.shape[1]
    #bins = P.shape[2]

    # Create fake data for testing purposes
    #    array = np.random.randint(0,2,(trials,bins))
    #    sigma = 45

    #Define kernel
    kernel_param = kernel_width/bin_size
    sigma = kernel_param/6.
    edges = np.arange(-3*sigma,3*sigma+1,1)
    kernel = sp.stats.norm.pdf(edges,0, sigma) #Use a gaussian function
    kernel = kernel/sum(kernel)
    
    #Compute Spike Density Function for all trials
    #X = np.zeros((trials,units,bins))
    X = sp.signal.convolve(P, kernel[None,:,None], mode='same',method='direct')

    print('Run time for SDF function was ' + str(round(time.time()-start)) + 'seconds') # toc
    return X

In [ ]:
#%% RUN
import os
import json
import shutil
import time
import numpy as np
import pandas as pd
from scipy.special.orthogonal import chebyc
from tqdm import tqdm
import pickle as pkl
import matplotlib
import matplotlib.pyplot as plt
from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache
#%%
data_directory = NP_DATA # must be updated to a valid directory in your filesystem

manifest_path = os.path.join(NEUROPIXELS_CACHE, "manifest.json")   # AllenSDK cache (was under data_directory)
cache = EcephysProjectCache.from_warehouse(manifest=manifest_path)
sessions = cache.get_session_table()
probes = cache.get_probes()
channels = cache.get_channels()
units = cache.get_units()
#%%
sess_id = 791319847
#%%
data = cache.get_session_data(sess_id)
#%%
class A_Config:
    def __init__(self, save_dir, sess_id, stimulus):
        
        #self.data_dir = data_dir
        self.save_dir = save_dir
        self.sess_id = sess_id
        self.stimulus = stimulus

        self.name = "{}/ID{}_{}".format(self.save_dir, self.sess_id, self.stimulus)

#%%
stim_label='natural_scenes'
allencon = A_Config(data_directory,sess_id,stim_label)
stim_label_list = ['natural_scenes','static_gratings','gabors','flashes']
bin_size=0.01
kernel_width=0.1
# X={}
# P={}
# for i, stim_label in enumerate(stim_label_list):
#     print(i)
#     allencon = A_Config(data_directory,sess_id,stim_label)
#     P[stim_label] = pkl.load(open(allencon.name+'_bin_{}_P.p'.format(bin_size),'rb'))
#     X[stim_label] = smooth_spikes(P[stim_label].values,kernel_width,bin_size)


In [ ]:
import os
import pickle as pkl
import time
class A_Config:
    def __init__(self, save_dir, sess_id, stimulus):
        
        #self.data_dir = data_dir
        self.save_dir = save_dir
        self.sess_id = sess_id
        self.stimulus = stimulus

        self.name = "{}/ID{}_{}".format(self.save_dir, self.sess_id, self.stimulus)
        
alpha = 0.05#float(sys.argv[1])

thresh_list = [0.75,0.70,0.65,0.60,0.55,0.50,0.45,0.40,0.35,0.30,0.25,0.20,0.15,0.10,0.05,0.01] #float(sys.argv[2])

#for idx0 in range(4):
idx0 = 0
stim_label_list = ['natural_scenes','static_gratings','gabors','flashes']
stim_label = stim_label_list[idx0]
ls = 60
if idx0==0:
    idxlist=range(ls)#[54]#[45, 47, 49, 51, 52, 54, 55, 57, 59]
elif idx0==1:
    idxlist=range(ls)#[2,3]#[0,2,3,4,5,6,7,8,9]
elif idx0==2:
    idxlist=range(ls)#[1]#[0,1,2]
else:
    ls = 3
    idxlist=range(ls)#[14]#[10,11,12,13,14,15,16,17,18,19]
#idx = 10
#idx = 0
print(stim_label)
def calc_cits(idx):
    sess_id = 791319847
    niter = 50
    method_name = 'tsci'#'tsci'#'gc'#'gc'#'neuropc'
    data_dir =  NP_DATA# 'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\data'#'/homes/rbiswas1/babaice_data/''C:\\Users\\BN Laptop\\OneDrive - UW\\research\\projects_git\\mice-aibs\\data'#
    results_dir =  NP_SAVE #'D:\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'+method_name#/homes/rbiswas1/babaice_save2/''C:\\Users\\BN Laptop\\OneDrive - UW\\research\\projects_git\\mice-aibs\\save\\'#
    #explist = {0:['N-5_T-300'],1:['N-5_T-300_noise-small','N-5_T-300_noise-medium','N-5_T-300_noise-strong'],2:['N-5_T-300_weak','N-5_T-300_medium','N-5_T-300_strong'],3:['N-5_T-300_weak','N-5_T-300_medium','N-5_T-300_strong'],4:['N-5_T-300'],5:['N-5_T-300'],6:['N-12_T-4600'],7:['N-5_T-2000'],8:['N-5_T-250'],9:['N-5_T-2000'],10:['N-5_T-250']}
    bin_size=0.01
    kernel_width=0.1
    allencon = A_Config(data_dir,sess_id,stim_label)
    allencon_save = A_Config(results_dir,sess_id,stim_label)
    subsampsize = 250

    results={}
    results_file = allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(bin_size,alpha,idx,niter)#results_dir+'%s.pkl' % (save_name)

    #if not os.path.exists(results_file):
    name = allencon.name+'_bin_{}_X_idx-{}.p'.format(bin_size,idx)
    units2use = pkl.load(open(allencon.name+'_units2use_stim_{}.p'.format(stim_label),'rb'))
    data = pkl.load(open(name,'rb'))
    print(data.shape)
    data = data[:,units2use]
    data -= data.mean(axis=0)
    data /= data.std(axis=0)
    data1 = data.T#np.row_stack((data,data+2*np.mean(data),data+3*np.mean(data),data+4*np.mean(data)))
    start_time = time.time()
    lag = 1
    print("shape of data "+str(data1.shape))
    print("T: "+str(data.shape[0])+' duplexed: '+str(data1.shape[0]))
    if method_name == 'neuropc':
        val_matrix_threshlist, causaleff, causaleff2 = val_neuropc(data1,lag,subsampsize=subsampsize,n_iter=niter,alpha=alpha,thresh_list=thresh_list)
    elif method_name == 'gc':
        val_matrix_threshlist, causaleff, causaleff2 = val_gc(data1,lag,alpha)
    elif method_name == 'tsci':
        val_matrix_threshlist, causaleff = cits_full_weighted(data1,lag,alpha, cond_dep = 'cond_dep_pcorr', thresh = 100)
        #causaleff = val_matrix_threshlist
        causaleff2 = causaleff
    runtimes=time.time() - start_time
    print(runtimes)
    results['runtimes'] = runtimes#np.array(runtimes).tolist()
    results['adj']=val_matrix_threshlist#outadj[thresh]#np.array(outadj).tolist()
    results['ce']=causaleff#outce[thresh]#np.array(outce).tolist()
    results['ce2']=causaleff2
    results['pvals']=1-val_matrix_threshlist#pvals[thresh]
    return results

    # Save data
    # with open(results_file, 'wb') as file1:
    #     pkl.dump(results, file1)
    # else:
    #     print("done before")
    #for idx in idxlist:
    #pool = multiprocessing.Pool(30)
    #out= pool.map(calc_cits, range(ls))

In [ ]:
out = calc_cits(0)

In [ ]:
out['ce'].shape

In [ ]:
import numpy as np

# find triples (x,y,z) where (x,y) and (x,z) are high but (y,z) is zero
ce = out['ce']
abs_ce = np.abs(ce)
nonzero_vals = abs_ce[abs_ce > 0]
thr = np.quantile(nonzero_vals, 0.90) if nonzero_vals.size > 0 else 0.0
print("threshold (90th pct of nonzero weights):", thr)

n = ce.shape[0]
triples = []
for x in range(n):
    for y in range(n):
        if x == y:
            continue
        if abs_ce[x, y] < thr:
            continue
        for z in range(n):
            if z == x or z == y:
                continue
            if abs_ce[x, z] < thr:
                continue
            if not np.isclose(ce[y, z], 0.0):
                continue
            triples.append((x, y, z, ce[x, y], ce[x, z], ce[y, z]))

triples_sorted = sorted(triples, key=lambda t: min(abs(t[3]), abs(t[4])), reverse=True)
print("Found triples (x,y) & (x,z) high, (y,z) zero:", len(triples_sorted))
for t in triples_sorted[:20]:
    x, y, z, wxy, wxz, wyz = t
    print(f"(x,y,z)=({x},{y},{z})  w(x,y)={wxy:.4f}  w(x,z)={wxz:.4f}  w(y,z)={wyz:.4f}")
    if 'units_idx_array' in globals():
        try:
            gid_x = units_idx_array[x]
            gid_y = units_idx_array[y]
            gid_z = units_idx_array[z]
            print(f"  global ids: x->{gid_x}, y->{gid_y}, z->{gid_z}")
        except Exception:
            pass

# -----------------------------------------------------------------------------
# correlation-based triple search: all pairwise correlations must be high
# -----------------------------------------------------------------------------
idx_for_corr = 0
bin_size = 0.01
corr_data_dir = NP_DATA
allencon = A_Config(corr_data_dir, sess_id, stim_label)
name = allencon.name + '_bin_{}_X_idx-{}.p'.format(bin_size, idx_for_corr)
raw = pkl.load(open(name, 'rb'))
mask_obj = pkl.load(open(allencon.name + '_units2use_stim_{}.p'.format(stim_label), 'rb'))
mask_arr = np.asarray(mask_obj)

if mask_arr.dtype == bool:
    units_idx_array = np.where(mask_arr)[0]
    data_std = raw[:, mask_arr]
else:
    units_idx_array = mask_arr
    data_std = raw[:, units_idx_array]

data_std = data_std.astype(float)
col_mean = data_std.mean(axis=0)
col_std = data_std.std(axis=0)
col_std[col_std == 0] = 1.0
data_std = (data_std - col_mean) / col_std

corr = np.corrcoef(data_std, rowvar=False)
abs_corr = np.abs(corr)
np.fill_diagonal(abs_corr, 0.0)
upper = abs_corr[np.triu_indices_from(abs_corr, k=1)]
nonzero_corr = upper[upper > 0]
corr_thr = np.quantile(nonzero_corr, 0.90) if nonzero_corr.size > 0 else 0.0
print("corr threshold (90th pct of |corr|>0):", corr_thr)

corr_triples = []
num_units = corr.shape[0]
for x in range(num_units - 2):
    for y in range(x + 1, num_units - 1):
        if abs_corr[x, y] < corr_thr:
            continue
        for z in range(y + 1, num_units):
            if abs_corr[x, z] < corr_thr or abs_corr[y, z] < corr_thr:
                continue
            corr_triples.append((x, y, z, corr[x, y], corr[x, z], corr[y, z]))

corr_triples_sorted = sorted(
    corr_triples,
    key=lambda t: min(abs(t[3]), abs(t[4]), abs(t[5])),
    reverse=True,
)

print("Triples with all pairwise correlations above threshold:", len(corr_triples_sorted))
for t in corr_triples_sorted[:20]:
    x, y, z, cxy, cxz, cyz = t
    print(
        f"(x,y,z)=({x},{y},{z})  corr(x,y)={cxy:.3f}  corr(x,z)={cxz:.3f}  corr(y,z)={cyz:.3f}"
    )
    print(
        f"  global ids: x->{units_idx_array[x]}, y->{units_idx_array[y]}, z->{units_idx_array[z]}"
    )

# -----------------------------------------------------------------------------
# intersection: weight criterion + correlation criterion simultaneously
# -----------------------------------------------------------------------------
combo_triples = []
for x, y, z, wxy, wxz, wyz in triples_sorted:
    if (
        abs_corr[x, y] >= corr_thr
        and abs_corr[x, z] >= corr_thr
        and abs_corr[y, z] >= corr_thr
    ):
        combo_triples.append(
            (
                x,
                y,
                z,
                wxy,
                wxz,
                wyz,
                corr[x, y],
                corr[x, z],
                corr[y, z],
            )
        )

print("Triples satisfying both weight+zero criterion and correlation criterion:", len(combo_triples))
for t in combo_triples[:20]:
    x, y, z, wxy, wxz, wyz, cxy, cxz, cyz = t
    print(
        f"(x,y,z)=({x},{y},{z})  w(x,y)={wxy:.4f}  w(x,z)={wxz:.4f}  w(y,z)={wyz:.4f}"
        f"  corr(x,y)={cxy:.3f}  corr(x,z)={cxz:.3f}  corr(y,z)={cyz:.3f}"
    )
    print(
        f"  global ids: x->{units_idx_array[x]}, y->{units_idx_array[y]}, z->{units_idx_array[z]}"
    )

In [ ]:
# choose which triple to inspect for raw time-series and scatter plots
if combo_triples:
    x, y, z = combo_triples[0][:3]
    print(f"Using top combined triple (x,y,z)=({x},{y},{z})")
else:
    raise ValueError("No combo_triples available; rerun upstream cell or adjust thresholds.")

In [ ]:
# Search for triples with lagged correlation pattern:
# - CFC constraints: x→y edge exists, x→z edge exists, y and z NOT connected
# - Strong lagged: x(t)↔y(t+1), y(t)↔z(t+1), z(t)↔x(t+1)
# - Conditional independence: y(t+1) ⊥ z(t+1) | x(t)
# - Weak synchronous: x(t)↔y(t), y(t)↔z(t), z(t)↔x(t)

import numpy as np
from scipy.stats import pearsonr

# Load CFC results (causal effect matrix)
# Assumes 'out' from calc_cits has been computed, or load from saved file
if 'out' not in globals() or 'ce' not in out:
    # Try to load from saved file
    results_dir = NP_SAVE
    allencon_save = A_Config(results_dir, sess_id, stim_label)
    bin_size_cfc = 0.01
    alpha = 0.05
    idx_cfc = 0
    niter = 50
    results_file = allencon_save.name + '_bin{}_alpha{}_idx{}_niter{}.p'.format(bin_size_cfc, alpha, idx_cfc, niter)
    if os.path.exists(results_file):
        results = pkl.load(open(results_file, 'rb'))
        ce = results['ce'] if isinstance(results, dict) and 'ce' in results else results
        print(f"Loaded CFC from: {results_file}")
    else:
        raise ValueError("CFC results not found. Please run calc_cits first or ensure results file exists.")
else:
    ce = out['ce']
    print("Using CFC from 'out' variable")

print(f"CFC matrix shape: {ce.shape}")
print(f"CFC matrix: {np.sum(ce != 0)} non-zero edges out of {ce.size} total")

# Load time series data for all units
idx_for_corr = 0
bin_size = 0.01
corr_data_dir = NP_DATA
allencon = A_Config(corr_data_dir, sess_id, stim_label)
name = allencon.name + '_bin_{}_X_idx-{}.p'.format(bin_size, idx_for_corr)
raw = pkl.load(open(name, 'rb'))
mask_obj = pkl.load(open(allencon.name + '_units2use_stim_{}.p'.format(stim_label), 'rb'))
mask_arr = np.asarray(mask_obj)

if mask_arr.dtype == bool:
    units_idx_array = np.where(mask_arr)[0]
    data_std = raw[:, mask_arr]
else:
    units_idx_array = mask_arr
    data_std = raw[:, units_idx_array]

# Standardize
data_std = data_std.astype(float)
col_mean = data_std.mean(axis=0)
col_std = data_std.std(axis=0)
col_std[col_std == 0] = 1.0
data_std = (data_std - col_mean) / col_std

print(f"Loaded data shape: {data_std.shape}")
print(f"Number of units: {data_std.shape[1]}")

# Parameters for search
lag = 1
lagged_corr_threshold = 0.3  # Minimum absolute lagged correlation
sync_corr_max = 0.4  # Maximum absolute synchronous correlation (should be weak)
cond_corr_max = 0.2  # Maximum absolute conditional correlation (for independence)
window_width = 0.3  # For conditional correlation windows
min_points_per_window = 60  # Minimum points needed in each window

def compute_lagged_corr(series_a, series_b, lag=1):
    """Compute correlation between series_a(t) and series_b(t+lag)"""
    if lag >= len(series_a) or lag >= len(series_b):
        return np.nan, 0
    a_trimmed = series_a[:-lag]
    b_shifted = series_b[lag:]
    if len(a_trimmed) < 2 or np.std(a_trimmed) == 0 or np.std(b_shifted) == 0:
        return np.nan, 0
    r, _ = pearsonr(a_trimmed, b_shifted)
    return r, len(a_trimmed)

def compute_sync_corr(series_a, series_b):
    """Compute synchronous correlation between series_a(t) and series_b(t)"""
    if len(series_a) != len(series_b) or len(series_a) < 2:
        return np.nan
    if np.std(series_a) == 0 or np.std(series_b) == 0:
        return np.nan
    r, _ = pearsonr(series_a, series_b)
    return r

def compute_conditional_corr(series_x, series_y_future, series_z_future, window_width=0.3, min_points=60, lag=1, n_intervals=3):
    """
    Compute y(t+1) vs z(t+1) correlation conditioned on x(t) in narrow windows.
    For conditional independence, we check the top n_intervals with smallest correlations.
    Returns the maximum absolute correlation from these top intervals.
    
    Approach:
    1. Create overlapping windows of x(t) values (width=window_width, step=width/2)
    2. For each window, compute y(t+1) vs z(t+1) correlation
    3. Sort intervals by |correlation| (smallest first)
    4. Take top n_intervals with smallest correlations
    5. Return max(|corr|) from these top intervals
    """
    if len(series_x) < 2 or len(series_y_future) < 2 or len(series_z_future) < 2:
        return np.nan, []
    
    # x(t) should align with y(t+1) and z(t+1), so trim x
    x_trimmed = series_x[:-lag] if len(series_x) > lag else series_x
    min_len = min(len(x_trimmed), len(series_y_future), len(series_z_future))
    x_trimmed = x_trimmed[:min_len]
    y_future = series_y_future[:min_len]
    z_future = series_z_future[:min_len]
    
    if len(x_trimmed) < min_points:
        return np.nan, []
    
    step = window_width / 2  # 50% overlap between windows
    centers = np.arange(x_trimmed.min(), x_trimmed.max() + step, step)
    interval_stats = []
    
    for center in centers:
        low = center - window_width / 2
        high = center + window_width / 2
        mask = (x_trimmed >= low) & (x_trimmed < high)
        count = mask.sum()
        
        if count < min_points:
            continue
            
        y_slice = y_future[mask]
        z_slice = z_future[mask]
        
        if len(y_slice) < 2 or np.std(y_slice) == 0 or np.std(z_slice) == 0:
            continue
            
        r, _ = pearsonr(y_slice, z_slice)
        
        interval_stats.append({
            'low': low,
            'high': high,
            'count': count,
            'r': r,
        })
    
    if len(interval_stats) < n_intervals:
        return np.nan, interval_stats
    
    # Sort by absolute correlation (smallest first) to find intervals showing independence
    interval_stats_sorted = sorted(interval_stats, key=lambda rec: abs(rec['r']))
    
    # Take the top n_intervals with smallest correlations
    top_intervals = interval_stats_sorted[:n_intervals]
    
    # Return the maximum absolute correlation from these top intervals
    # (if independence holds, even the "worst" of the top intervals should be small)
    max_abs_corr = max(abs(rec['r']) for rec in top_intervals)
    
    return max_abs_corr, interval_stats

# Search through all triples
n_units = data_std.shape[1]
lagged_triples = []

print(f"\nSearching for triples with:")
print(f"  - CFC: x→y edge exists (ce[x,y] != 0)")
print(f"  - CFC: x→z edge exists (ce[x,z] != 0)")
print(f"  - CFC: y and z NOT connected (ce[y,z] == 0 and ce[z,y] == 0)")
print(f"  - Lagged correlations (x(t)↔y(t+1), etc.) >= {lagged_corr_threshold}")
print(f"  - Synchronous correlations (x(t)↔y(t), etc.) <= {sync_corr_max}")
print(f"  - Conditional correlation y(t+1)↔z(t+1)|x(t) <= {cond_corr_max}")
print(f"\nSearching through {n_units} units...")

for x in range(n_units):
    if (x + 1) % 10 == 0:
        print(f"  Progress: {x+1}/{n_units} units checked...")
    
    series_x = data_std[:, x]
    
    for y in range(n_units):
        if x == y:
            continue
        
        # CFC constraint: x→y edge must exist
        if np.isclose(ce[x, y], 0.0):
            continue
        
        series_y = data_std[:, y]
        
        # Check synchronous correlation x(t) vs y(t) - should be weak
        sync_xy = compute_sync_corr(series_x, series_y)
        if np.isnan(sync_xy) or abs(sync_xy) > sync_corr_max:
            continue
        
        # Check lagged correlation x(t) vs y(t+1) - should be strong
        lagged_xy, n_xy = compute_lagged_corr(series_x, series_y, lag)
        if np.isnan(lagged_xy) or abs(lagged_xy) < lagged_corr_threshold:
            continue
        
        for z in range(n_units):
            if z == x or z == y:
                continue
            
            # CFC constraints: x→z edge must exist, and y-z must NOT be connected (either direction)
            if np.isclose(ce[x, z], 0.0):  # x→z edge must exist
                continue
            # y and z must NOT be connected in either direction
            if not np.isclose(ce[y, z], 0.0) or not np.isclose(ce[z, y], 0.0):
                continue
            
            series_z = data_std[:, z]
            
            # Check synchronous correlations - should all be weak
            sync_xz = compute_sync_corr(series_x, series_z)
            sync_yz = compute_sync_corr(series_y, series_z)
            if (np.isnan(sync_xz) or abs(sync_xz) > sync_corr_max or
                np.isnan(sync_yz) or abs(sync_yz) > sync_corr_max):
                continue
            
            # Check lagged correlations - should all be strong
            lagged_yz, n_yz = compute_lagged_corr(series_y, series_z, lag)
            lagged_zx, n_zx = compute_lagged_corr(series_z, series_x, lag)
            if (np.isnan(lagged_yz) or abs(lagged_yz) < lagged_corr_threshold or
                np.isnan(lagged_zx) or abs(lagged_zx) < lagged_corr_threshold):
                continue
            
            # Check conditional independence: y(t+1) ⊥ z(t+1) | x(t)
            # Need y(t+1) and z(t+1) series
            y_future = series_y[lag:] if len(series_y) > lag else series_y
            z_future = series_z[lag:] if len(series_z) > lag else series_z
            
            max_cond_corr, interval_stats = compute_conditional_corr(
                series_x, y_future, z_future, window_width, min_points_per_window, lag, n_intervals=3
            )
            
            if np.isnan(max_cond_corr) or max_cond_corr > cond_corr_max:
                continue
            
            # All criteria satisfied!
            lagged_triples.append({
                'x': x, 'y': y, 'z': z,
                'lagged_xy': lagged_xy,
                'lagged_yz': lagged_yz,
                'lagged_zx': lagged_zx,
                'sync_xy': sync_xy,
                'sync_xz': sync_xz,
                'sync_yz': sync_yz,
                'max_cond_corr': max_cond_corr,
                'interval_stats': interval_stats,
                'global_x': units_idx_array[x],
                'global_y': units_idx_array[y],
                'global_z': units_idx_array[z],
            })

# Sort by average lagged correlation strength
lagged_triples_sorted = sorted(
    lagged_triples,
    key=lambda t: (abs(t['lagged_xy']) + abs(t['lagged_yz']) + abs(t['lagged_zx'])) / 3,
    reverse=True
)

print(f"\n{'='*80}")
print(f"Found {len(lagged_triples_sorted)} triples satisfying all criteria!")
print(f"{'='*80}\n")

if lagged_triples_sorted:
    print("Top triples (sorted by average lagged correlation strength):\n")
    for i, t in enumerate(lagged_triples_sorted[:20], 1):
        print(f"{i}. (x,y,z) = ({t['x']},{t['y']},{t['z']})")
        print(f"   Global IDs: x->{t['global_x']}, y->{t['global_y']}, z->{t['global_z']}")
        print(f"   Lagged correlations:")
        print(f"     x(t)↔y(t+1): {t['lagged_xy']:.3f}")
        print(f"     y(t)↔z(t+1): {t['lagged_yz']:.3f}")
        print(f"     z(t)↔x(t+1): {t['lagged_zx']:.3f}")
        print(f"   Synchronous correlations (weak):")
        print(f"     x(t)↔y(t): {t['sync_xy']:.3f}")
        print(f"     x(t)↔z(t): {t['sync_xz']:.3f}")
        print(f"     y(t)↔z(t): {t['sync_yz']:.3f}")
        print(f"   Conditional correlation y(t+1)↔z(t+1)|x(t): {t['max_cond_corr']:.3f}")
        print()
else:
    print("No triples found. Try adjusting thresholds:")
    print(f"  - Lower lagged_corr_threshold (currently {lagged_corr_threshold})")
    print(f"  - Raise sync_corr_max (currently {sync_corr_max})")
    print(f"  - Raise cond_corr_max (currently {cond_corr_max})")


In [ ]:
# Visualize a selected triple from lagged_triples_sorted
# This creates a figure showing lagged correlations and conditional independence

import matplotlib.pyplot as plt
import os

if 'lagged_triples_sorted' not in globals() or not lagged_triples_sorted:
    raise ValueError("No lagged triples found. Run the previous cell first.")

# Select the top triple (or change index to visualize a different one)
selected_idx = 0
if selected_idx >= len(lagged_triples_sorted):
    selected_idx = 0

t = lagged_triples_sorted[selected_idx]
x_local, y_local, z_local = t['x'], t['y'], t['z']
x_global, y_global, z_global = t['global_x'], t['global_y'], t['global_z']

print(f"Visualizing triple #{selected_idx+1}: (x,y,z) = ({x_local},{y_local},{z_local})")
print(f"Global IDs: x->{x_global}, y->{y_global}, z->{z_global}")

# Load time series for this triple
selected_global = [x_global, y_global, z_global]
series_dict, loaded_path = get_time_series_for_global_indices(selected_global, sess_id, stim_label, idx_for_corr)

valid_series = []
valid_globals = []
for global_idx in selected_global:
    ts = series_dict.get(global_idx)
    if ts is None:
        raise ValueError(f"Time series missing for global unit {global_idx}; cannot plot.")
    valid_series.append(ts)
    valid_globals.append(global_idx)

xi, yi, zi = valid_series  # x, y, z time series

# Create lagged series
lag = 1
xi_t = xi[:-lag]
yi_t = yi[:-lag]
zi_t = zi[:-lag]
yi_t1 = yi[lag:]
zi_t1 = zi[lag:]
xi_t1 = xi[lag:]  # for z(t) vs x(t+1)

# Create figure with 2 rows x 3 columns
fig_rows, fig_cols = 2, 3
panel_w, panel_h = 4.3, 3.4
fig_width, fig_height = panel_w * fig_cols, panel_h * fig_rows
fig, axes = plt.subplots(fig_rows, fig_cols, figsize=(fig_width, fig_height), dpi=150)
top_axes = axes[0]
bottom_axes = axes[1]

# Top row: Lagged correlations
lagged_pairs = [
    (xi_t, yi_t1, f"x(t) vs y(t+1)\nr={t['lagged_xy']:.3f}"),
    (yi_t, zi_t1, f"y(t) vs z(t+1)\nr={t['lagged_yz']:.3f}"),
    (zi_t, xi_t1, f"z(t) vs x(t+1)\nr={t['lagged_zx']:.3f}"),
]

for ax, (xs, ys, title) in zip(top_axes, lagged_pairs):
    ax.scatter(xs, ys, s=6, alpha=0.4, color='#4c78a8')
    slope, intercept = np.polyfit(xs, ys, 1)
    x_line = np.linspace(xs.min(), xs.max(), 200)
    y_line = slope * x_line + intercept
    
    residuals = ys - (slope * xs + intercept)
    dof = max(len(xs) - 2, 1)
    s_err = np.sqrt(np.sum(residuals ** 2) / dof)
    x_mean = xs.mean()
    sxx = np.sum((xs - x_mean) ** 2)
    if sxx == 0:
        band = np.full_like(x_line, s_err)
    else:
        band = s_err * np.sqrt(1 / len(xs) + (x_line - x_mean) ** 2 / sxx)
    ax.fill_between(x_line, y_line - 1.96 * band, y_line + 1.96 * band,
                    color='#f58518', alpha=0.15, linewidth=0)
    ax.plot(x_line, y_line, color='#f58518', linewidth=2)
    ax.set_title(title, fontsize=11)
    ax.grid(color='0.85', linestyle='--', linewidth=0.6, alpha=0.6)

# Bottom row: Conditional correlations y(t+1) vs z(t+1) | x(t)
# Use the interval_stats from the search
interval_stats_sorted = sorted(t['interval_stats'], key=lambda rec: abs(rec['r']))
bottom_intervals = interval_stats_sorted[:3]

for ax in bottom_axes[len(bottom_intervals):]:
    ax.set_axis_off()

# Align x(t) with y(t+1) and z(t+1)
x_cond = xi[:-lag]
y_cond = yi[lag:]
z_cond = zi[lag:]
min_len = min(len(x_cond), len(y_cond), len(z_cond))
x_cond = x_cond[:min_len]
y_cond = y_cond[:min_len]
z_cond = z_cond[:min_len]

for ax, rec in zip(bottom_axes, bottom_intervals):
    mask = (x_cond >= rec['low']) & (x_cond < rec['high'])
    y_slice = y_cond[mask]
    z_slice = z_cond[mask]
    
    if len(y_slice) < 2:
        ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center')
        ax.set_axis_off()
        continue
    
    ax.scatter(y_slice, z_slice, s=10, alpha=0.45, color='#4c78a8')
    slope, intercept = np.polyfit(y_slice, z_slice, 1)
    x_line = np.linspace(y_slice.min(), y_slice.max(), 200)
    y_line = slope * x_line + intercept
    
    residuals = z_slice - (slope * y_slice + intercept)
    dof = max(len(y_slice) - 2, 1)
    s_err = np.sqrt(np.sum(residuals ** 2) / dof)
    x_mean = y_slice.mean()
    sxx = np.sum((y_slice - x_mean) ** 2)
    if sxx == 0:
        band = np.full_like(x_line, s_err)
    else:
        band = s_err * np.sqrt(1 / len(y_slice) + (x_line - x_mean) ** 2 / sxx)
    ax.fill_between(x_line, y_line - 1.96 * band, y_line + 1.96 * band,
                    color='#f58518', alpha=0.15, linewidth=0)
    ax.plot(x_line, y_line, color='#f58518', linewidth=2)
    
    title = f"y(t+1) vs z(t+1) | x(t)∈[{rec['low']:.2f},{rec['high']:.2f}]\ncount={rec['count']} r={rec['r']:.3f}"
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(f"Unit {y_global} (y at t+1)")
    ax.set_ylabel(f"Unit {z_global} (z at t+1)")
    ax.grid(color='0.85', linestyle='--', linewidth=0.6, alpha=0.6)

fig.suptitle(
    f"Lagged Correlation Pattern: (x,y,z)=({x_local},{y_local},{z_local})\n"
    f"Strong lagged: x(t)↔y(t+1), y(t)↔z(t+1), z(t)↔x(t+1) | "
    f"Weak sync: x(t)↔y(t)={t['sync_xy']:.3f}, x(t)↔z(t)={t['sync_xz']:.3f}, y(t)↔z(t)={t['sync_yz']:.3f}",
    fontsize=14, fontweight='bold', y=0.98
)
fig.subplots_adjust(hspace=0.4, wspace=0.25)

# Save figure
output_dir = _outdir('fig4_motifs/legacy/figs/panels')
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(
    output_dir,
    f"lagged_triple_{x_local}_{y_local}_{z_local}_sixpanel.png"
)
fig.savefig(output_path, dpi=400, bbox_inches='tight')
print(f"Saved lagged correlation figure to {output_path}")
plt.show()


In [ ]:
# load_time_series_for_indices.py
# Assumes A_Config, pkl, np, matplotlib already available in the notebook environment.

def load_trial_data_and_unit_map(sess_id, stim_label, idx,
                                 bin_size=0.01, data_dir=NP_DATA):
    """
    Load the saved X matrix and the units2use mask for a given session/stimulus/index.
    Returns:
      data: (T, n_selected_units) standardized (zero mean, unit std)
      units_idx: array of global unit indices (indices in session 'units' list) that correspond to columns of `data`
      raw_path: path to the loaded pkl (for debugging)
    """
    allencon = A_Config(data_dir, sess_id, stim_label)
    name = allencon.name + '_bin_{}_X_idx-{}.p'.format(bin_size, idx)
    raw = pkl.load(open(name, 'rb'))         # raw shape: (T, n_total_units) in saved file
    mask = pkl.load(open(allencon.name + '_units2use_stim_{}.p'.format(stim_label), 'rb'))
    units_idx = np.where(mask)[0] if mask is not None and mask.dtype == bool else np.asarray(mask)

    data = raw[:, mask] if mask is not None and mask.dtype == bool else raw[:, units_idx]

    # standardize columns (match calc_cits preprocessing)
    data = data.astype(float)
    col_mean = data.mean(axis=0)
    col_std = data.std(axis=0)
    col_std[col_std == 0] = 1.0
    data = (data - col_mean) / col_std

    return data, units_idx, name

def get_time_series_for_global_indices(global_indices, sess_id, stim_label, idx=0,
                                       bin_size=0.01, data_dir=NP_DATA):
    """
    For a list of global unit indices (indices into the full session units array),
    load the trial data and return a dict mapping global_index -> 1D time series.
    """
    data, units_idx, path = load_trial_data_and_unit_map(sess_id, stim_label, idx,
                                                         bin_size=bin_size, data_dir=data_dir)
    units_idx = np.array(units_idx)
    series = {}
    for g in global_indices:
        matches = np.where(units_idx == g)[0]
        if matches.size == 0:
            series[g] = None
        else:
            col = matches[0]
            series[g] = data[:, col]
    return series, path

# Use the selected triple from the combined criteria block
selected_local = (x, y, z)
idx = idx_for_corr  # reuse same trial index as correlation computation
allencon = A_Config(NP_DATA, sess_id, stim_label)
mask_or_idx = pkl.load(open(allencon.name + '_units2use_stim_{}.p'.format(stim_label),'rb'))
mask_arr = np.asarray(mask_or_idx)

if mask_arr.dtype == bool:
    units_idx_array = np.where(mask_arr)[0]
else:
    units_idx_array = mask_arr

selected_global = list(units_idx_array[list(selected_local)])
series_dict, loaded_path = get_time_series_for_global_indices(selected_global, sess_id, stim_label, idx)

print("Loaded from:", loaded_path)
for label, local_idx, global_idx in zip(['x','y','z'], selected_local, selected_global):
    ts = series_dict.get(global_idx)
    if ts is None:
        print(f"Local {label}={local_idx} global {global_idx}: NOT FOUND in this trial's units2use")
    else:
        print(f"Local {label}={local_idx} global {global_idx}: time series length = {ts.shape[0]}")

# Scatter plots for each pair (x,y), (x,z), (y,z)
valid_series = []
valid_globals = []
for global_idx in selected_global:
    ts = series_dict.get(global_idx)
    if ts is None:
        raise ValueError(f"Time series missing for global unit {global_idx}; cannot plot.")
    valid_series.append(ts)
    valid_globals.append(global_idx)

pairs = [(0, 1, 'x vs y'), (0, 2, 'x vs z'), (1, 2, 'y vs z')]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (i, j, title) in zip(axes, pairs):
    xi = valid_series[i]
    xj = valid_series[j]
    ax.scatter(xi, xj, s=6, alpha=0.4, color='#4c78a8')
    slope, intercept = np.polyfit(xi, xj, 1)
    x_line = np.linspace(xi.min(), xi.max(), 200)
    y_line = slope * x_line + intercept

    residuals = xj - (slope * xi + intercept)
    dof = max(len(xi) - 2, 1)
    s_err = np.sqrt(np.sum(residuals ** 2) / dof)
    x_mean = xi.mean()
    sxx = np.sum((xi - x_mean) ** 2)
    if sxx == 0:
        band = np.full_like(x_line, s_err)
    else:
        band = s_err * np.sqrt(1 / len(xi) + (x_line - x_mean) ** 2 / sxx)
    ax.fill_between(x_line, y_line - 1.96 * band, y_line + 1.96 * band,
                    color='#f58518', alpha=0.15, linewidth=0)
    ax.plot(x_line, y_line, color='#f58518', linewidth=2)
    ax.set_xlabel(f"Unit {valid_globals[i]} (local {selected_local[i]})")
    ax.set_ylabel(f"Unit {valid_globals[j]} (local {selected_local[j]})")
    corr_val = np.corrcoef(xi, xj)[0, 1]
    ax.set_title(f"{title}\nPearson r={corr_val:.3f}")
fig.suptitle(f"Scatter plots for triple (x,y,z)=({selected_local[0]}, {selected_local[1]}, {selected_local[2]})")
plt.tight_layout()
plt.show()

In [ ]:
# conditional_yz_correlations.py
import numpy as np

if 'valid_series' not in globals():
    raise ValueError("Run the time-series loading/scatter cell first to populate valid_series.")

xi, yi, zi = valid_series  # xi=x-series, yi=y-series, zi=z-series
window_width = 0.3
step = window_width / 2
min_points = 60
centers = np.arange(xi.min(), xi.max() + step, step)

interval_stats = []
for center in centers:
    low = center - window_width / 2
    high = center + window_width / 2
    mask = (xi >= low) & (xi < high)
    count = mask.sum()
    if count < min_points:
        continue
    y_slice = yi[mask]
    z_slice = zi[mask]
    if y_slice.size == 0 or z_slice.size == 0:
        continue
    if np.std(y_slice) == 0 or np.std(z_slice) == 0:
        continue
    r = np.corrcoef(y_slice, z_slice)[0, 1]
    interval_stats.append({
        'low': low,
        'high': high,
        'center': center,
        'count': count,
        'r': r,
    })

if not interval_stats:
    print("No intervals met the min_points requirement; adjust window_width or min_points.")
else:
    interval_stats_sorted = sorted(interval_stats, key=lambda rec: abs(rec['r']))
    print(f"Top intervals (window_width={window_width}, min_points={min_points}) with smallest |corr(y,z)|:")
    for rec in interval_stats_sorted[:10]:
        print(f"x in [{rec['low']:.2f}, {rec['high']:.2f}]  count={rec['count']}  corr(y,z)={rec['r']:.3f}")

    num_plots = min(3, len(interval_stats_sorted))
    fig, axes = plt.subplots(1, num_plots, figsize=(5 * num_plots, 4))
    if num_plots == 1:
        axes = [axes]
    for ax, rec in zip(axes, interval_stats_sorted[:num_plots]):
        mask = (xi >= rec['low']) & (xi < rec['high'])
        y_slice = yi[mask]
        z_slice = zi[mask]
        ax.scatter(y_slice, z_slice, s=12, alpha=0.5, color='#4c78a8')
        slope, intercept = np.polyfit(y_slice, z_slice, 1)
        x_line = np.linspace(y_slice.min(), y_slice.max(), 200)
        y_line = slope * x_line + intercept

        residuals = z_slice - (slope * y_slice + intercept)
        dof = max(len(y_slice) - 2, 1)
        s_err = np.sqrt(np.sum(residuals ** 2) / dof)
        x_mean = y_slice.mean()
        sxx = np.sum((y_slice - x_mean) ** 2)
        if sxx == 0:
            band = np.full_like(x_line, s_err)
        else:
            band = s_err * np.sqrt(1 / len(y_slice) + (x_line - x_mean) ** 2 / sxx)
        ax.fill_between(x_line, y_line - 1.96 * band, y_line + 1.96 * band,
                        color='#f58518', alpha=0.15, linewidth=0)
        ax.plot(x_line, y_line, color='#f58518', linewidth=2)
        ax.set_title(f"x∈[{rec['low']:.2f},{rec['high']:.2f}]\ncount={rec['count']} r={rec['r']:.3f}")
        ax.set_xlabel(f"Unit {selected_global[1]} (y)")
        ax.set_ylabel(f"Unit {selected_global[2]} (z)")
    fig.suptitle("y vs z correlations within narrow x windows")
    plt.tight_layout()
    plt.show()

In [ ]:
# combined_panel_save.py
import os
import numpy as np
import matplotlib.pyplot as plt

def _plot_with_band(ax, xs, ys, xlabel, ylabel, title, scatter_alpha=0.35, scatter_size=8):
    if xs.size < 2 or ys.size < 2:
        ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center')
        ax.set_axis_off()
        return
    ax.scatter(xs, ys, s=scatter_size, alpha=scatter_alpha, color='#4c78a8')
    slope, intercept = np.polyfit(xs, ys, 1)
    x_line = np.linspace(xs.min(), xs.max(), 200)
    y_line = slope * x_line + intercept

    residuals = ys - (slope * xs + intercept)
    dof = max(len(xs) - 2, 1)
    s_err = np.sqrt(np.sum(residuals ** 2) / dof)
    x_mean = xs.mean()
    sxx = np.sum((xs - x_mean) ** 2)
    if sxx == 0:
        band = np.full_like(x_line, s_err)
    else:
        band = s_err * np.sqrt(1 / len(xs) + (x_line - x_mean) ** 2 / sxx)
    ax.fill_between(x_line, y_line - 1.96 * band, y_line + 1.96 * band,
                    color='#f58518', alpha=0.15, linewidth=0)
    ax.plot(x_line, y_line, color='#f58518', linewidth=2)
    corr_val = np.corrcoef(xs, ys)[0, 1]
    ax.set_title(f"{title}\nPearson r={corr_val:.3f}", fontsize=11, pad=8)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.tick_params(labelsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(color='0.85', linestyle='--', linewidth=0.6, alpha=0.6)

if 'valid_series' not in globals():
    raise ValueError("Run the scatter-plot cell to populate valid_series before creating the combined panel.")
if 'interval_stats_sorted' not in locals():
    raise ValueError("Run the conditional_yz_correlations cell to populate interval stats before creating the combined panel.")
if not interval_stats_sorted:
    raise ValueError("No conditional intervals available; cannot create six-panel figure.")

xi, yi, zi = valid_series
pairs = [(0, 1, 'x vs y'), (0, 2, 'x vs z'), (1, 2, 'y vs z')]
fig_rows, fig_cols = 2, 3
panel_w, panel_h = 4.3, 3.4
fig_width, fig_height = panel_w * fig_cols, panel_h * fig_rows
fig, axes = plt.subplots(fig_rows, fig_cols, figsize=(fig_width, fig_height), dpi=150)
top_axes = axes[0]
bottom_axes = axes[1]

for ax, (i, j, title) in zip(top_axes, pairs):
    xlabel = f"Unit {valid_globals[i]} (local {selected_local[i]})"
    ylabel = f"Unit {valid_globals[j]} (local {selected_local[j]})"
    _plot_with_band(ax, valid_series[i], valid_series[j], xlabel, ylabel, title, scatter_alpha=0.3, scatter_size=6)

bottom_intervals = interval_stats_sorted[:3]
for ax in bottom_axes[len(bottom_intervals):]:
    ax.set_axis_off()
for ax, rec in zip(bottom_axes, bottom_intervals):
    mask = (xi >= rec['low']) & (xi < rec['high'])
    y_slice = yi[mask]
    z_slice = zi[mask]
    title = f"y vs z | x∈[{rec['low']:.2f},{rec['high']:.2f}]\ncount={rec['count']} r={rec['r']:.3f}"
    _plot_with_band(
        ax, y_slice, z_slice,
        xlabel=f"Unit {selected_global[1]} (y)",
        ylabel=f"Unit {selected_global[2]} (z)",
        title=title,
        scatter_alpha=0.45,
        scatter_size=10,
    )

fig.suptitle(
    f"Triple (x,y,z)=({selected_local[0]}, {selected_local[1]}, {selected_local[2]})",
    fontsize=18, fontweight='bold', y=0.98
)
fig.subplots_adjust(hspace=0.4, wspace=0.25)

output_dir = _outdir('fig4_motifs/legacy/figs/panels')
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(
    output_dir,
    f"triple_{selected_local[0]}_{selected_local[1]}_{selected_local[2]}_sixpanel.png"
)
fig.savefig(output_path, dpi=400, bbox_inches='tight')
print(f"Saved six-panel figure to {output_path}")
plt.show()

In [ ]:
# lagged_six_panel_variants.py
import numpy as np
import matplotlib.pyplot as plt
import os

if '_lag_pair' not in globals():
    def _lag_pair(series_a, series_b, lag=0):
        if lag == 0:
            return series_a, series_b
        if lag > 0:
            if lag >= len(series_a) or lag >= len(series_b):
                raise ValueError(f"Lag {lag} too large for series of length {len(series_a)}")
            return series_a[:-lag], series_b[lag:]
        lag_abs = abs(lag)
        if lag_abs >= len(series_a) or lag_abs >= len(series_b):
            raise ValueError(f"Lag {lag} too large for series of length {len(series_a)}")
        return series_a[lag_abs:], series_b[:-lag_abs]

if '_trim_mask_for_lag' not in globals():
    def _trim_mask_for_lag(mask, lag=0):
        if mask is None or lag == 0:
            return mask
        if lag > 0:
            return mask[:-lag]
        lag_abs = abs(lag)
        return mask[lag_abs:]

def _lag_label(lag):
    if lag == 0:
        return "t vs t"
    if lag > 0:
        return f"t vs t+{lag}"
    return f"t vs t{lag}"

def _make_six_panel_lag(pair_list, lag, suffix, title_note):
    if 'valid_series' not in globals():
        raise ValueError("valid_series missing; run the scatter cell first.")
    if 'interval_stats_sorted' not in globals() or not interval_stats_sorted:
        raise ValueError("interval statistics missing; run the conditional correlations cell first.")
    xi, yi, zi = valid_series
    fig_rows, fig_cols = 2, 3
    panel_w, panel_h = 4.3, 3.4
    fig_width, fig_height = panel_w * fig_cols, panel_h * fig_rows
    fig, axes = plt.subplots(fig_rows, fig_cols, figsize=(fig_width, fig_height), dpi=150)
    top_axes = axes[0]
    bottom_axes = axes[1]
    lag_desc = _lag_label(lag)
    for ax, (i, j, title) in zip(top_axes, pair_list):
        xs, ys = _lag_pair(valid_series[i], valid_series[j], lag)
        if xs.size < 2 or ys.size < 2:
            ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center')
            ax.set_axis_off()
            continue
        lag_title = f"{title} | {lag_desc}"
        xlabel = f"Unit {valid_globals[i]} (local {selected_local[i]})"
        ylabel = f"Unit {valid_globals[j]} (local {selected_local[j]})"
        _plot_with_band(ax, xs, ys, xlabel, ylabel, lag_title, scatter_alpha=0.35, scatter_size=7)
    remaining = len(top_axes) - len(pair_list)
    if remaining > 0:
        for ax in top_axes[-remaining:]:
            ax.set_axis_off()

    bottom_intervals = interval_stats_sorted[:3]
    for ax in bottom_axes[len(bottom_intervals):]:
        ax.set_axis_off()
    y_aligned, z_aligned = _lag_pair(yi, zi, lag)
    for ax, rec in zip(bottom_axes, bottom_intervals):
        mask_x = (xi >= rec['low']) & (xi < rec['high'])
        mask_trim = _trim_mask_for_lag(mask_x, lag)
        count = int(mask_trim.sum()) if mask_trim is not None else 0
        if mask_trim is None or count < 2:
            ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center')
            ax.set_axis_off()
            continue
        y_slice = y_aligned[mask_trim]
        z_slice = z_aligned[mask_trim]
        subtitle = (
            f"y vs z | x∈[{rec['low']:.2f},{rec['high']:.2f}]\n"
            f"{lag_desc}, count={count}"
        )
        _plot_with_band(
            ax, y_slice, z_slice,
            xlabel=f"Unit {selected_global[1]} (y)",
            ylabel=f"Unit {selected_global[2]} (z)",
            title=subtitle + title_note,
            scatter_alpha=0.45,
            scatter_size=10,
        )

    fig.suptitle(
        f"Triple (x,y,z)=({selected_local[0]}, {selected_local[1]}, {selected_local[2]})\n"
        f"Lagged panel: {lag_desc}{title_note}",
        fontsize=18,
        fontweight='bold',
        y=0.98,
    )
    fig.subplots_adjust(hspace=0.4, wspace=0.25)
    output_dir = _outdir('fig4_motifs/legacy/figs/panels')
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(
        output_dir,
        f"triple_{selected_local[0]}_{selected_local[1]}_{selected_local[2]}_{suffix}.png",
    )
    fig.savefig(output_path, dpi=400, bbox_inches='tight')
    print(f"Saved lagged six-panel figure to {output_path}")
    plt.show()

pairs_default = [(0, 1, 'x vs y'), (0, 2, 'x vs z'), (1, 2, 'y vs z')]
pairs_swapped = [(1, 0, 'y vs x'), (1, 2, 'y vs z'), (0, 2, 'x vs z')]
lag_variants = [
    {
        'pair_list': pairs_default,
        'lag': 1,
        'suffix': 'sixpanel_x_t_vs_future_y',
        'title_note': ' (second variable at t+1)',
    },
    {
        'pair_list': pairs_swapped,
        'lag': 1,
        'suffix': 'sixpanel_y_t_vs_future_x',
        'title_note': ' (swapped axes, second variable at t+1)',
    },
 ]
for variant in lag_variants:
    _make_six_panel_lag(**variant)

In [ ]:
# quadruple_conditional_search.py
import numpy as np
from itertools import combinations

if 'out' not in globals():
    raise ValueError("Run the causal inference cell that populates `out` before searching for quadruples.")

ce = out['ce']
abs_ce = np.abs(ce)
nonzero_weights = abs_ce[abs_ce > 0]
n = ce.shape[0]

sess_id_local = globals().get('sess_id', 791319847)
stim_label_local = globals().get('stim_label', 'natural_scenes')
idx_for_corr_local = globals().get('idx_for_corr', 0)
bin_size_local = globals().get('bin_size', 0.01)
data_dir_local = NP_DATA

def load_standardized_trial():
    allencon = A_Config(data_dir_local, sess_id_local, stim_label_local)
    name = allencon.name + '_bin_{}_X_idx-{}.p'.format(bin_size_local, idx_for_corr_local)
    raw = pkl.load(open(name, 'rb'))
    mask_obj = pkl.load(open(allencon.name + '_units2use_stim_{}.p'.format(stim_label_local), 'rb'))
    mask_arr = np.asarray(mask_obj)
    if mask_arr.dtype == bool:
        units_idx = np.where(mask_arr)[0]
        data = raw[:, mask_arr]
    else:
        units_idx = mask_arr
        data = raw[:, units_idx]
    data = data.astype(float)
    col_mean = data.mean(axis=0)
    col_std = data.std(axis=0)
    col_std[col_std == 0] = 1.0
    data = (data - col_mean) / col_std
    return data, units_idx

if 'data_std' not in globals() or data_std.shape[1] != ce.shape[0]:
    data_std, units_idx_array = load_standardized_trial()
    print("Reloaded standardized time series for correlation checks.")
else:
    units_idx_array = np.asarray(units_idx_array)

corr = np.corrcoef(data_std, rowvar=False)
abs_corr = np.abs(corr)
np.fill_diagonal(abs_corr, 0.0)
upper = abs_corr[np.triu_indices_from(abs_corr, k=1)]
nonzero_corr = upper[upper > 0]

# default values (will be overwritten per sweep)
window_width = 0.3
half_width = window_width / 2
joint_min_points = 80
single_min_points = 80
low_joint_max = 0.20
single_low_limit = 0.30
max_results = 5
max_centers = 15

def make_centers(series):
    s_min, s_max = series.min(), series.max()
    if np.isclose(s_min, s_max):
        return np.array([s_min])
    return np.linspace(s_min, s_max, max_centers)

def find_joint_window(a_vals, d_vals, b_vals, c_vals):
    centers_b = make_centers(b_vals)
    centers_c = make_centers(c_vals)
    for cb in centers_b:
        mask_b = (b_vals >= cb - half_width) & (b_vals < cb + half_width)
        if mask_b.sum() < joint_min_points:
            continue
        for cc in centers_c:
            mask = mask_b & (c_vals >= cc - half_width) & (c_vals < cc + half_width)
            count = mask.sum()
            if count < joint_min_points:
                continue
            a_slice = a_vals[mask]
            d_slice = d_vals[mask]
            if a_slice.size < 2 or d_slice.size < 2:
                continue
            corr_val = np.corrcoef(a_slice, d_slice)[0, 1]
            if np.isnan(corr_val):
                continue
            if abs(corr_val) <= low_joint_max:
                return {
                    'b_center': cb,
                    'c_center': cc,
                    'b_range': (cb - half_width, cb + half_width),
                    'c_range': (cc - half_width, cc + half_width),
                    'count': count,
                    'corr': corr_val,
                    'abs_corr': abs(corr_val),
                }
    return None

def scan_single(series, a_vals, d_vals):
    centers = make_centers(series)
    best = None
    for center in centers:
        mask = (series >= center - half_width) & (series < center + half_width)
        count = mask.sum()
        if count < single_min_points:
            continue
        a_slice = a_vals[mask]
        d_slice = d_vals[mask]
        if a_slice.size < 2 or d_slice.size < 2:
            continue
        corr_val = np.corrcoef(a_slice, d_slice)[0, 1]
        if np.isnan(corr_val):
            continue
        abs_corr_val = abs(corr_val)
        if best is None or abs_corr_val < best['abs_corr']:
            best = {
                'center': center,
                'range': (center - half_width, center + half_width),
                'count': count,
                'corr': corr_val,
                'abs_corr': abs_corr_val,
            }
    return best

def evaluate_quadruple(a_idx, b_idx, c_idx, d_idx):
    a_vals = data_std[:, a_idx]
    b_vals = data_std[:, b_idx]
    c_vals = data_std[:, c_idx]
    d_vals = data_std[:, d_idx]
    joint_hit = find_joint_window(a_vals, d_vals, b_vals, c_vals)
    if joint_hit is None:
        return None
    b_single = scan_single(b_vals, a_vals, d_vals)
    c_single = scan_single(c_vals, a_vals, d_vals)
    if b_single is None or c_single is None:
        return None
    if b_single['abs_corr'] <= single_low_limit or c_single['abs_corr'] <= single_low_limit:
        return None
    return {
        'a': a_idx,
        'b': b_idx,
        'c': c_idx,
        'd': d_idx,
        'joint': joint_hit,
        'b_single': b_single,
        'c_single': c_single,
        'corr_ad_full': corr[a_idx, d_idx],
        'weights': {
            'b_to_a': ce[b_idx, a_idx],
            'c_to_d': ce[c_idx, d_idx],
            'b_to_c': ce[b_idx, c_idx],
            'c_to_b': ce[c_idx, b_idx],
        }
    }

def pairs_have_high_corr(nodes):
    for i, j in combinations(nodes, 2):
        if np.isnan(abs_corr[i, j]) or abs_corr[i, j] < corr_pair_cutoff:
            return False
    return True

param_sweeps = [
    {
        'label': 'baseline',
        'weight_quantile': 0.80,
        'corr_quantile': 0.90,
        'window_width': 0.30,
        'joint_min_points': 80,
        'single_min_points': 80,
        'low_joint_max': 0.25,
        'single_low_limit': 0.30,
    },
    {
        'label': 'moderate',
        'weight_quantile': 0.70,
        'corr_quantile': 0.85,
        'window_width': 0.30,
        'joint_min_points': 70,
        'single_min_points': 70,
        'low_joint_max': 0.30,
        'single_low_limit': 0.28,
    },
    {
        'label': 'broad',
        'weight_quantile': 0.60,
        'corr_quantile': 0.80,
        'window_width': 0.35,
        'joint_min_points': 60,
        'single_min_points': 60,
        'low_joint_max': 0.35,
        'single_low_limit': 0.25,
    },
    {
        'label': 'max-relaxed',
        'weight_quantile': 0.50,
        'corr_quantile': 0.75,
        'window_width': 0.40,
        'joint_min_points': 50,
        'single_min_points': 50,
        'low_joint_max': 0.40,
        'single_low_limit': 0.22,
    }
]

best_results = None
best_params = None

for sweep_idx, cfg in enumerate(param_sweeps, 1):
    window_width = cfg['window_width']
    half_width = window_width / 2
    joint_min_points = cfg['joint_min_points']
    single_min_points = cfg['single_min_points']
    low_joint_max = cfg['low_joint_max']
    single_low_limit = cfg['single_low_limit']
    weight_quantile = cfg['weight_quantile']
    corr_quantile = cfg['corr_quantile']

    weight_thr = np.quantile(nonzero_weights, weight_quantile) if nonzero_weights.size else 0.0
    corr_pair_cutoff = np.quantile(nonzero_corr, corr_quantile) if nonzero_corr.size else 0.0
    corr_pair_cutoff = max(corr_pair_cutoff, 0.10)
    outgoing_targets = [np.where(abs_ce[row] > weight_thr)[0] for row in range(n)]

    print(
        f"Sweep {sweep_idx} ({cfg['label']}): weight>Q{weight_quantile:.2f} (~{weight_thr:.4f}), "
        f"corr>Q{corr_quantile:.2f} (~{corr_pair_cutoff:.3f}), joint≤{low_joint_max:.2f}, "
        f"single≥{single_low_limit:.2f}, window={window_width:.2f}"
    )

    results = []
    seen = set()
    for b_idx in range(n):
        for a_idx in outgoing_targets[b_idx]:
            if a_idx == b_idx:
                continue
            for c_idx in range(n):
                if c_idx in (a_idx, b_idx):
                    continue
                if not outgoing_targets[c_idx].size:
                    continue
                if not (abs_ce[b_idx, c_idx] > weight_thr or abs_ce[c_idx, b_idx] > weight_thr):
                    continue
                for d_idx in outgoing_targets[c_idx]:
                    if d_idx in (a_idx, b_idx, c_idx):
                        continue
                    quad_key = (a_idx, b_idx, c_idx, d_idx)
                    if quad_key in seen:
                        continue
                    seen.add(quad_key)
                    nodes = (a_idx, b_idx, c_idx, d_idx)
                    if not pairs_have_high_corr(nodes):
                        continue
                    eval_result = evaluate_quadruple(a_idx, b_idx, c_idx, d_idx)
                    if eval_result is None:
                        continue
                    results.append(eval_result)
                    print(f"  Found candidate quadruple (a,b,c,d)=({a_idx},{b_idx},{c_idx},{d_idx})")
                    if len(results) >= max_results:
                        break
                if len(results) >= max_results:
                    break
            if len(results) >= max_results:
                break
        if len(results) >= max_results:
            break

    if results:
        best_results = results
        best_params = {**cfg, 'weight_thr': weight_thr, 'corr_pair_cutoff': corr_pair_cutoff}
        break
    else:
        print(f"  → No quadruples for sweep {sweep_idx} ({cfg['label']}).\n")

if not best_results:
    print("No quadruples met the criteria even after sweeping thresholds.")
else:
    print(
        f"\nFound {len(best_results)} quadruple(s) using sweep '{best_params['label']}' "
        f"(weight_thr≈{best_params['weight_thr']:.4f}, corr_cutoff≈{best_params['corr_pair_cutoff']:.3f})."
    )
    for idx, rec in enumerate(best_results, 1):
        a_idx, b_idx, c_idx, d_idx = rec['a'], rec['b'], rec['c'], rec['d']
        a_gid, b_gid, c_gid, d_gid = (
            units_idx_array[a_idx],
            units_idx_array[b_idx],
            units_idx_array[c_idx],
            units_idx_array[d_idx],
        )
        print(f"[{idx}] (a,b,c,d)=({a_idx},{b_idx},{c_idx},{d_idx})  globals=({a_gid},{b_gid},{c_gid},{d_gid})")
        print(
            f"     weights: b→a={rec['weights']['b_to_a']:.4f}, c→d={rec['weights']['c_to_d']:.4f}, "
            f"b→c={rec['weights']['b_to_c']:.4f}, c→b={rec['weights']['c_to_b']:.4f}"
        )
        print(f"     corr(a,d) full={rec['corr_ad_full']:.3f}")
        joint = rec['joint']
        print(
            f"     joint window: b∈[{joint['b_range'][0]:.2f},{joint['b_range'][1]:.2f}], "
            f"c∈[{joint['c_range'][0]:.2f},{joint['c_range'][1]:.2f}], "
            f"count={joint['count']} corr(a,d)={joint['corr']:.3f}"
        )
        b_single = rec['b_single']
        print(
            f"     b-only min |corr|={b_single['abs_corr']:.3f} "
            f"(center={b_single['center']:.2f}, count={b_single['count']})"
        )
        c_single = rec['c_single']
        print(
            f"     c-only min |corr|={c_single['abs_corr']:.3f} "
            f"(center={c_single['center']:.2f}, count={c_single['count']})"
        )

In [ ]:
# quadruple_conditioned_panel_plot.py
import numpy as np
import os
import matplotlib.pyplot as plt

if 'best_results' not in globals() or not best_results:
    raise ValueError("Run the quadruple search cell first to populate `best_results`.")
if '_plot_with_band' not in globals():
    def _plot_with_band(ax, xs, ys, xlabel, ylabel, title, scatter_alpha=0.35, scatter_size=8):
        if xs.size < 2 or ys.size < 2:
            ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center')
            ax.set_axis_off()
            return
        ax.scatter(xs, ys, s=scatter_size, alpha=scatter_alpha, color='#4c78a8')
        slope, intercept = np.polyfit(xs, ys, 1)
        x_line = np.linspace(xs.min(), xs.max(), 200)
        y_line = slope * x_line + intercept

        residuals = ys - (slope * xs + intercept)
        dof = max(len(xs) - 2, 1)
        s_err = np.sqrt(np.sum(residuals ** 2) / dof)
        x_mean = xs.mean()
        sxx = np.sum((xs - x_mean) ** 2)
        if sxx == 0:
            band = np.full_like(x_line, s_err)
        else:
            band = s_err * np.sqrt(1 / len(xs) + (x_line - x_mean) ** 2 / sxx)
        ax.fill_between(x_line, y_line - 1.96 * band, y_line + 1.96 * band,
                        color='#f58518', alpha=0.15, linewidth=0)
        ax.plot(x_line, y_line, color='#f58518', linewidth=2)
        corr_val = np.corrcoef(xs, ys)[0, 1]
        ax.set_title(f"{title}\nPearson r={corr_val:.3f}", fontsize=10, pad=8)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.tick_params(labelsize=8)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(color='0.85', linestyle='--', linewidth=0.6, alpha=0.6)

if 'data_std' not in globals():
    raise ValueError("Standardized trial data (`data_std`) not found; rerun the loading cell.")
if 'units_idx_array' not in globals():
    raise ValueError("`units_idx_array` not found; rerun the correlation setup cell.")

selected_quad = best_results[0]
quad_indices = {
    'a': selected_quad['a'],
    'b': selected_quad['b'],
    'c': selected_quad['c'],
    'd': selected_quad['d'],
}
series_map = {label: data_std[:, idx] for label, idx in quad_indices.items()}
global_ids = {label: int(units_idx_array[idx]) for label, idx in quad_indices.items()}
weights = selected_quad['weights']

pairs = [
    ('a', 'b'), ('a', 'c'), ('a', 'd'),
    ('b', 'c'), ('b', 'd'), ('c', 'd'),
]
pair_rows = [pairs[i:i + 3] for i in range(0, len(pairs), 3)]

def _make_bins(values, width=0.3, step=0.25, min_count=60, max_bins=3):
    rng = values.max() - values.min()
    if rng == 0:
        return [(values.min(), values.max(), len(values))]
    centers = np.arange(values.min(), values.max() + step, step)
    bins = []
    for center in centers:
        low = center - width / 2
        high = center + width / 2
        mask = (values >= low) & (values < high)
        if mask.sum() >= min_count:
            bins.append((low, high, mask.sum()))
        if len(bins) >= max_bins:
            break
    if not bins:
        bins.append((values.min(), values.max(), len(values)))
    while len(bins) < max_bins:
        bins.append(bins[-1])
    return bins[:max_bins]

def _mask_from_bounds(values, low, high):
    return (values >= low) & (values < high)

def _ranges_overlap(r1, r2, tol=0.05):
    low1, high1 = r1
    low2, high2 = r2
    overlap = min(high1, high2) - max(low1, low2)
    span = min(high1 - low1, high2 - low2)
    return overlap > tol * span

def _windows_overlap(cand, jb):
    return (_ranges_overlap((cand[0], cand[1]), (jb[0], jb[1])) and
            _ranges_overlap((cand[2], cand[3]), (jb[2], jb[3])))

def _is_duplicate(cand, existing, tol=1e-3):
    for jb in existing:
        if (
            abs(cand[0] - jb[0]) <= tol
            and abs(cand[1] - jb[1]) <= tol
            and abs(cand[2] - jb[2]) <= tol
            and abs(cand[3] - jb[3]) <= tol
        ):
            return True
    return False

def _generate_joint_bins(series_map, selected_quad, min_points=35, target_bins=3):
    b_vals = series_map['b']
    c_vals = series_map['c']
    def _count_window(b_low, b_high, c_low, c_high):
        mask = _mask_from_bounds(b_vals, b_low, b_high) & _mask_from_bounds(c_vals, c_low, c_high)
        return int(mask.sum())
    window_widths = [0.25, 0.35, 0.5, 0.7, 0.9, 1.2]
    grid_points = 12
    joint_candidates = []
    for window_width in window_widths:
        half_w = window_width / 2
        candidate_batch = []
        for cb in np.linspace(b_vals.min(), b_vals.max(), grid_points):
            for cc in np.linspace(c_vals.min(), c_vals.max(), grid_points):
                b_low, b_high = cb - half_w, cb + half_w
                c_low, c_high = cc - half_w, cc + half_w
                count = _count_window(b_low, b_high, c_low, c_high)
                if count >= min_points:
                    candidate_batch.append((b_low, b_high, c_low, c_high, count))
        joint_candidates.extend(candidate_batch)
        if len(candidate_batch) >= target_bins:
            break
    def _fallback_window():
        joint_info = selected_quad.get('joint', {}) if isinstance(selected_quad, dict) else {}
        b_range = joint_info.get('b_range', (float(b_vals.min()), float(b_vals.max())))
        c_range = joint_info.get('c_range', (float(c_vals.min()), float(c_vals.max())))
        b_low, b_high = map(float, b_range)
        c_low, c_high = map(float, c_range)
        count = _count_window(b_low, b_high, c_low, c_high)
        if count == 0:
            count = len(b_vals)
        return (b_low, b_high, c_low, c_high, int(count))
    if not joint_candidates:
        joint_candidates.append(_fallback_window())
    joint_candidates.sort(key=lambda item: -item[4])
    joint_bins = []
    for cand in joint_candidates:
        if all(not _windows_overlap(cand, jb) for jb in joint_bins):
            joint_bins.append(cand)
        if len(joint_bins) >= target_bins:
            break
    if len(joint_bins) < target_bins:
        for cand in joint_candidates:
            if len(joint_bins) >= target_bins:
                break
            if _is_duplicate(cand, joint_bins):
                continue
            joint_bins.append(cand)
    if len(joint_bins) < target_bins:
        fallback = _fallback_window()
        b_center = 0.5 * (fallback[0] + fallback[1])
        c_center = 0.5 * (fallback[2] + fallback[3])
        base_width_b = max(fallback[1] - fallback[0], 0.1)
        base_width_c = max(fallback[3] - fallback[2], 0.1)
        for expand_factor in (1.2, 1.6, 2.0, 2.6):
            b_half = base_width_b * expand_factor / 2
            c_half = base_width_c * expand_factor / 2
            b_low, b_high = b_center - b_half, b_center + b_half
            c_low, c_high = c_center - c_half, c_center + c_half
            count = _count_window(b_low, b_high, c_low, c_high)
            if count == 0:
                continue
            widened = (b_low, b_high, c_low, c_high, int(count))
            if not _is_duplicate(widened, joint_bins):
                joint_bins.append(widened)
            if len(joint_bins) >= target_bins:
                break
    if not joint_bins:
        joint_bins = [joint_candidates[0]]
    while len(joint_bins) < target_bins:
        joint_bins.append(joint_bins[-1])
    return joint_bins[:target_bins]

mask_all = np.ones_like(series_map['a'], dtype=bool)
a_idx = quad_indices['a']
b_idx = quad_indices['b']
c_idx = quad_indices['c']
d_idx = quad_indices['d']

b_bins = _make_bins(series_map['b'], width=0.25, step=0.12, min_count=40, max_bins=3)
c_bins = _make_bins(series_map['c'], width=0.25, step=0.12, min_count=40, max_bins=3)

joint_bins = _generate_joint_bins(series_map, selected_quad, min_points=35, target_bins=3)

fig_rows = len(pair_rows) + 3
fig_cols = 3
fig, axes = plt.subplots(fig_rows, fig_cols, figsize=(4.3 * fig_cols, 3.4 * fig_rows), dpi=150)
axes = np.atleast_2d(axes)

row_idx = 0
for row_pairs in pair_rows:
    row_axes = axes[row_idx]
    for ax, (left_label, right_label) in zip(row_axes, row_pairs):
        mask = mask_all
        xs = series_map[left_label][mask]
        ys = series_map[right_label][mask]
        xlabel = f"{left_label.upper()} (unit {global_ids[left_label]})"
        ylabel = f"{right_label.upper()} (unit {global_ids[right_label]})"
        title = f"{left_label.upper()}-{right_label.upper()} | full sample (n={xs.size})"
        _plot_with_band(ax, xs, ys, xlabel, ylabel, title, scatter_alpha=0.3, scatter_size=7)
    for ax in row_axes[len(row_pairs):]:
        ax.set_axis_off()
    axes[row_idx, 0].text(-0.3, 0.5, f"Full sample {row_idx + 1}", transform=axes[row_idx, 0].transAxes,
                         rotation=90, va='center', ha='center', fontsize=11, fontweight='bold')
    row_idx += 1

for label, bins in [('b', b_bins), ('c', c_bins)]:
    row_axes = axes[row_idx]
    for col_idx, (low, high, count) in enumerate(bins):
        ax = row_axes[col_idx]
        if label == 'b':
            mask = _mask_from_bounds(series_map['b'], low, high)
            subtitle = f"A-D | b∈[{low:.2f},{high:.2f}] (n={mask.sum()})"
            color_note = 'Condition on b'
        else:
            mask = _mask_from_bounds(series_map['c'], low, high)
            subtitle = f"A-D | c∈[{low:.2f},{high:.2f}] (n={mask.sum()})"
            color_note = 'Condition on c'
        xs = series_map['a'][mask]
        ys = series_map['d'][mask]
        if xs.size < 2 or ys.size < 2:
            ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center', fontsize=9)
            ax.set_axis_off()
        else:
            _plot_with_band(
                ax, xs, ys,
                f"A (unit {global_ids['a']})",
                f"D (unit {global_ids['d']})",
                subtitle,
                scatter_alpha=0.45,
                scatter_size=9,
            )
    axes[row_idx, 0].text(-0.3, 0.5, color_note, transform=axes[row_idx, 0].transAxes, rotation=90,
                         va='center', ha='center', fontsize=11, fontweight='bold')
    row_idx += 1

row_axes = axes[row_idx]
for col_idx, (b_low, b_high, c_low, c_high, count) in enumerate(joint_bins):
    ax = row_axes[col_idx]
    mask = _mask_from_bounds(series_map['b'], b_low, b_high) & _mask_from_bounds(series_map['c'], c_low, c_high)
    xs = series_map['a'][mask]
    ys = series_map['d'][mask]
    title = (
        f"A-D | b∈[{b_low:.2f},{b_high:.2f}],\n"
        f"c∈[{c_low:.2f},{c_high:.2f}] (n={mask.sum()})"
    )
    if xs.size < 2 or ys.size < 2:
        ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center', fontsize=9)
        ax.set_axis_off()
    else:
        _plot_with_band(
            ax, xs, ys,
            f"A (unit {global_ids['a']})",
            f"D (unit {global_ids['d']})",
            title,
            scatter_alpha=0.55,
            scatter_size=10,
        )
axes[row_idx, 0].text(-0.3, 0.5, 'Condition on b & c', transform=axes[row_idx, 0].transAxes, rotation=90,
                     va='center', ha='center', fontsize=11, fontweight='bold')

fig.suptitle(
    f"Quadruple (a,b,c,d)=({a_idx}, {b_idx}, {c_idx}, {d_idx})\n"
    f"Globals=({global_ids['a']}, {global_ids['b']}, {global_ids['c']}, {global_ids['d']})",
    fontsize=18,
    fontweight='bold',
    y=0.992,
 )
fig.tight_layout(rect=[0, 0, 1, 0.965])

output_dir = _outdir('fig4_motifs/legacy/figs/panels')
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(
    output_dir,
    f"quadruple_{a_idx}_{b_idx}_{c_idx}_{d_idx}_conditioned_panel.png",
)
fig.savefig(output_path, dpi=320, bbox_inches='tight')
print(f"Saved quadruple conditioned panel to {output_path}")
print(
    f"Weights: b→a={weights['b_to_a']:.4f}, c→d={weights['c_to_d']:.4f}, "
    f"b→c={weights['b_to_c']:.4f}, c→b={weights['c_to_b']:.4f}",
)

def corr_ad(mask):
    if mask.sum() < 3:
        return np.nan
    return float(np.corrcoef(series_map['a'][mask], series_map['d'][mask])[0, 1])

print("corr(a,d) summaries:")
print(f"  full: {corr_ad(mask_all):.3f} (n={mask_all.sum()})")
for label, bins in [('b', b_bins), ('c', c_bins)]:
    for (low, high, count) in bins:
        mask = _mask_from_bounds(series_map[label], low, high)
        print(f"  A-D | {label}∈[{low:.2f},{high:.2f}]: {corr_ad(mask):.3f} (n={mask.sum()})")
for (b_low, b_high, c_low, c_high, count) in joint_bins:
    mask = _mask_from_bounds(series_map['b'], b_low, b_high) & _mask_from_bounds(series_map['c'], c_low, c_high)
    print(
        f"  A-D | b∈[{b_low:.2f},{b_high:.2f}], c∈[{c_low:.2f},{c_high:.2f}]: "
        f"{corr_ad(mask):.3f} (n={mask.sum()})"
    )

In [ ]:
# lagged_quadruple_conditioned_panels.py
import numpy as np
import matplotlib.pyplot as plt
import os

if 'best_results' not in globals() or not best_results:
    raise ValueError("best_results missing; run the quadruple search cell first.")
if 'data_std' not in globals():
    raise ValueError("data_std missing; rerun the standardization cell.")
if '_plot_with_band' not in globals():
    raise ValueError("_plot_with_band helper missing; run earlier cells.")
if '_mask_from_bounds' not in globals():
    def _mask_from_bounds(values, low, high):
        return (values >= low) & (values < high)
if '_lag_pair' not in globals():
    def _lag_pair(series_a, series_b, lag=0):
        if lag == 0:
            return series_a, series_b
        if lag > 0:
            if lag >= len(series_a) or lag >= len(series_b):
                raise ValueError(f"Lag {lag} too large")
            return series_a[:-lag], series_b[lag:]
        lag_abs = abs(lag)
        if lag_abs >= len(series_a) or lag_abs >= len(series_b):
            raise ValueError(f"Lag {lag} too large")
        return series_a[lag_abs:], series_b[:-lag_abs]
if '_trim_mask_for_lag' not in globals():
    def _trim_mask_for_lag(mask, lag=0):
        if lag == 0:
            return mask
        if lag > 0:
            return mask[:-lag]
        lag_abs = abs(lag)
        return mask[lag_abs:]
if '_lag_label' not in globals():
    def _lag_label(lag):
        if lag == 0:
            return "t vs t"
        if lag > 0:
            return f"t vs t+{lag}"
        return f"t vs t{lag}"

def _generate_joint_bins_for_lag(series_map, selected_quad, lag, min_points=25, target_bins=3, distinct_b_thresh=0.65, min_final_points=110):
    b_vals = series_map['b']
    c_vals = series_map['c']
    b_min, b_max = float(b_vals.min()), float(b_vals.max())
    c_min, c_max = float(c_vals.min()), float(c_vals.max())

    def _count_window(b_low, b_high, c_low, c_high):
        mask = _mask_from_bounds(b_vals, b_low, b_high) & _mask_from_bounds(c_vals, c_low, c_high)
        mask_aligned = _trim_mask_for_lag(mask, lag)
        return int(mask_aligned.sum())

    def _fallback_window():
        joint_info = selected_quad.get('joint', {}) if isinstance(selected_quad, dict) else {}
        b_range = joint_info.get('b_range', (b_min, b_max))
        c_range = joint_info.get('c_range', (c_min, c_max))
        b_low, b_high = map(float, b_range)
        c_low, c_high = map(float, c_range)
        count = _count_window(b_low, b_high, c_low, c_high)
        if count == 0:
            count = max(len(b_vals) - abs(lag), 1)
        return (b_low, b_high, c_low, c_high, int(count))

    def _ranges_overlap(r1, r2, tol=0.05):
        low1, high1 = r1
        low2, high2 = r2
        overlap = min(high1, high2) - max(low1, low2)
        span = min(high1 - low1, high2 - low2)
        return overlap > tol * span

    def _windows_overlap(cand, jb):
        return (_ranges_overlap((cand[0], cand[1]), (jb[0], jb[1])) and
                _ranges_overlap((cand[2], cand[3]), (jb[2], jb[3])))

    def _is_duplicate(cand, existing, tol=1e-3):
        for jb in existing:
            if (abs(cand[0] - jb[0]) <= tol and
                    abs(cand[1] - jb[1]) <= tol and
                    abs(cand[2] - jb[2]) <= tol and
                    abs(cand[3] - jb[3]) <= tol):
                return True
        return False

    def _b_overlap_ratio(cand, jb):
        overlap = min(cand[1], jb[1]) - max(cand[0], jb[0])
        if overlap <= 0:
            return 0.0
        span = min(jb[1] - jb[0], cand[1] - cand[0])
        if span <= 0:
            return 0.0
        return overlap / span

    def _is_distinct_b(cand, existing, thresh):
        if not existing:
            return True
        for jb in existing:
            if _b_overlap_ratio(cand, jb) >= thresh:
                return False
        return True

    def _expand_to_min_points(bin_tuple, min_required):
        b_low, b_high, c_low, c_high, count = bin_tuple
        if count >= min_required:
            return bin_tuple
        b_center = 0.5 * (b_low + b_high)
        c_center = 0.5 * (c_low + c_high)
        base_width_b = max(b_high - b_low, 0.05)
        base_width_c = max(c_high - c_low, 0.05)
        expansion_factors = (1.2, 1.6, 2.0, 2.6, 3.2, 4.0)
        for factor in expansion_factors:
            b_half = 0.5 * base_width_b * factor
            c_half = 0.5 * base_width_c * factor
            new_b_low = max(b_min, b_center - b_half)
            new_b_high = min(b_max, b_center + b_half)
            new_c_low = max(c_min, c_center - c_half)
            new_c_high = min(c_max, c_center + c_half)
            expanded_count = _count_window(new_b_low, new_b_high, new_c_low, new_c_high)
            if expanded_count >= min_required:
                return (new_b_low, new_b_high, new_c_low, new_c_high, int(expanded_count))
        full_count = _count_window(b_min, b_max, c_min, c_max)
        return (b_min, b_max, c_min, c_max, int(full_count))

    window_widths = [0.25, 0.35, 0.5, 0.7, 0.9, 1.2]
    grid_points = 12
    joint_candidates = []
    for window_width in window_widths:
        half_w = window_width / 2
        candidate_batch = []
        for cb in np.linspace(b_vals.min(), b_vals.max(), grid_points):
            for cc in np.linspace(c_vals.min(), c_vals.max(), grid_points):
                b_low, b_high = cb - half_w, cb + half_w
                c_low, c_high = cc - half_w, cc + half_w
                count = _count_window(b_low, b_high, c_low, c_high)
                if count >= min_points:
                    candidate_batch.append((b_low, b_high, c_low, c_high, count))
        joint_candidates.extend(candidate_batch)
        if len(candidate_batch) >= target_bins:
            break

    if not joint_candidates:
        joint_candidates.append(_fallback_window())

    joint_candidates.sort(key=lambda item: -item[4])

    joint_bins = []
    for cand in joint_candidates:
        if not _is_distinct_b(cand, joint_bins, distinct_b_thresh):
            continue
        if all(not _windows_overlap(cand, jb) for jb in joint_bins):
            joint_bins.append(cand)
        if len(joint_bins) >= target_bins:
            break

    if len(joint_bins) < target_bins:
        for cand in joint_candidates:
            if len(joint_bins) >= target_bins:
                break
            if _is_duplicate(cand, joint_bins):
                continue
            joint_bins.append(cand)

    if len(joint_bins) < target_bins:
        fallback = _fallback_window()
        b_center = 0.5 * (fallback[0] + fallback[1])
        c_center = 0.5 * (fallback[2] + fallback[3])
        base_width_b = max(fallback[1] - fallback[0], 0.1)
        base_width_c = max(fallback[3] - fallback[2], 0.1)
        for expand_factor in (1.2, 1.6, 2.0, 2.6, 3.2):
            if len(joint_bins) >= target_bins:
                break
            b_half = base_width_b * expand_factor / 2
            c_half = base_width_c * expand_factor / 2
            for offset_dir in (-1.0, 0.0, 1.0):
                if len(joint_bins) >= target_bins:
                    break
                b_shift = offset_dir * base_width_b * 0.4
                b_low = b_center - b_half + b_shift
                b_high = b_center + b_half + b_shift
                c_low = c_center - c_half
                c_high = c_center + c_half
                count = _count_window(b_low, b_high, c_low, c_high)
                if count == 0:
                    continue
                widened = (b_low, b_high, c_low, c_high, int(count))
                if not _is_distinct_b(widened, joint_bins, distinct_b_thresh * 0.85):
                    continue
                if _is_duplicate(widened, joint_bins):
                    continue
                joint_bins.append(widened)

    if not joint_bins:
        joint_bins = [joint_candidates[0]]
    while len(joint_bins) < target_bins:
        joint_bins.append(joint_bins[-1])
    joint_bins = [_expand_to_min_points(jb, min_final_points) for jb in joint_bins[:target_bins]]
    return joint_bins

selected_quad = best_results[0]
quad_indices = {
    'a': selected_quad['a'],
    'b': selected_quad['b'],
    'c': selected_quad['c'],
    'd': selected_quad['d'],
}
series_map = {label: data_std[:, idx] for label, idx in quad_indices.items()}
global_ids = {label: int(units_idx_array[idx]) for label, idx in quad_indices.items()}
weights = selected_quad['weights']

pairs = [
    ('a', 'b'), ('a', 'c'), ('a', 'd'),
    ('b', 'c'), ('b', 'd'), ('c', 'd'),
]
pair_rows = [pairs[i:i + 3] for i in range(0, len(pairs), 3)]
pair_rows_swapped = [[(right, left) for (left, right) in row] for row in pair_rows]

b_bins = _make_bins(series_map['b'], width=0.25, step=0.12, min_count=40, max_bins=3)
c_bins = _make_bins(series_map['c'], width=0.25, step=0.12, min_count=40, max_bins=3)

def _render_quadruple_lagged(pair_rows_cfg, lag, suffix, title_note, ad_pair=('a', 'd')):
    joint_bins = _generate_joint_bins_for_lag(
        series_map,
        selected_quad,
        lag,
        min_points=30,
        target_bins=3,
        min_final_points=120,
    )
    lag_desc = _lag_label(lag)
    fig_rows = len(pair_rows_cfg) + 3
    fig_cols = 3
    panel_w, panel_h = 4.3, 3.4
    fig_width, fig_height = panel_w * fig_cols, panel_h * fig_rows
    fig, axes = plt.subplots(fig_rows, fig_cols, figsize=(fig_width, fig_height), dpi=150)
    axes = np.atleast_2d(axes)

    row_idx = 0
    for row_pairs in pair_rows_cfg:
        row_axes = axes[row_idx]
        for ax, (left_label, right_label) in zip(row_axes, row_pairs):
            xs, ys = _lag_pair(series_map[left_label], series_map[right_label], lag)
            if xs.size < 2 or ys.size < 2:
                ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center', fontsize=9)
                ax.set_axis_off()
                continue
            title = f"{left_label.upper()}-{right_label.upper()} | {lag_desc} (n={xs.size})"
            xlabel = f"{left_label.upper()} (unit {global_ids[left_label]})"
            ylabel = f"{right_label.upper()} (unit {global_ids[right_label]})"
            _plot_with_band(ax, xs, ys, xlabel, ylabel, title, scatter_alpha=0.3, scatter_size=7)
        for ax in row_axes[len(row_pairs):]:
            ax.set_axis_off()
        axes[row_idx, 0].text(-0.3, 0.5, f"Full sample {row_idx + 1}", transform=axes[row_idx, 0].transAxes,
                             rotation=90, va='center', ha='center', fontsize=11, fontweight='bold')
        row_idx += 1

    base_xs, base_ys = _lag_pair(series_map[ad_pair[0]], series_map[ad_pair[1]], lag)
    for label, bins in [('b', b_bins), ('c', c_bins)]:
        row_axes = axes[row_idx]
        for col_idx, (low, high, count) in enumerate(bins):
            ax = row_axes[col_idx]
            mask = _mask_from_bounds(series_map[label], low, high)
            mask_aligned = _trim_mask_for_lag(mask, lag)
            if mask_aligned.sum() < 2:
                ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center', fontsize=9)
                ax.set_axis_off()
                continue
            xs = base_xs[mask_aligned]
            ys = base_ys[mask_aligned]
            subtitle = f"{ad_pair[0].upper()}-{ad_pair[1].upper()} | {label}∈[{low:.2f},{high:.2f}]\n{lag_desc} (n={xs.size})"
            _plot_with_band(
                ax, xs, ys,
                f"{ad_pair[0].upper()} (unit {global_ids[ad_pair[0]]})",
                f"{ad_pair[1].upper()} (unit {global_ids[ad_pair[1]]})",
                subtitle,
                scatter_alpha=0.45,
                scatter_size=9,
            )
        axes[row_idx, 0].text(-0.3, 0.5, f"Condition on {label}", transform=axes[row_idx, 0].transAxes, rotation=90,
                             va='center', ha='center', fontsize=11, fontweight='bold')
        row_idx += 1

    row_axes = axes[row_idx]
    for col_idx, (b_low, b_high, c_low, c_high, count) in enumerate(joint_bins):
        ax = row_axes[col_idx]
        mask_bc = (_mask_from_bounds(series_map['b'], b_low, b_high) &
                   _mask_from_bounds(series_map['c'], c_low, c_high))
        mask_aligned = _trim_mask_for_lag(mask_bc, lag)
        if mask_aligned.sum() < 2:
            ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center', fontsize=9)
            ax.set_axis_off()
            continue
        xs = base_xs[mask_aligned]
        ys = base_ys[mask_aligned]
        title = (
            f"{ad_pair[0].upper()}-{ad_pair[1].upper()} | {lag_desc}\\n"
            f"b∈[{b_low:.2f},{b_high:.2f}], c∈[{c_low:.2f},{c_high:.2f}] (n={xs.size})"
        )
        _plot_with_band(
            ax, xs, ys,
            f"{ad_pair[0].upper()} (unit {global_ids[ad_pair[0]]})",
            f"{ad_pair[1].upper()} (unit {global_ids[ad_pair[1]]})",
            title,
            scatter_alpha=0.55,
            scatter_size=10,
        )
    axes[row_idx, 0].text(-0.3, 0.5, 'Condition on b & c', transform=axes[row_idx, 0].transAxes, rotation=90,
                         va='center', ha='center', fontsize=11, fontweight='bold')

    fig.suptitle(
        f"Quadruple (a,b,c,d)=({quad_indices['a']}, {quad_indices['b']}, {quad_indices['c']}, {quad_indices['d']})\\n"
        f"Lagged panel: {lag_desc}{title_note}",
        fontsize=18,
        fontweight='bold',
        y=0.99,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.965])
    output_dir = _outdir('fig4_motifs/legacy/figs/panels')
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(
        output_dir,
        f"quadruple_{quad_indices['a']}_{quad_indices['b']}_{quad_indices['c']}_{quad_indices['d']}_{suffix}.png",
    )
    fig.savefig(output_path, dpi=320, bbox_inches='tight')
    print(f"Saved lagged quadruple panel to {output_path}")
    plt.show()

lag_variants_quad = [
    {
        'pair_rows_cfg': pair_rows,
        'lag': 1,
        'suffix': 'lagpanel_a_t_vs_future_d',
        'title_note': ' (second variable at t+1)',
        'ad_pair': ('a', 'd'),
    },
    {
        'pair_rows_cfg': pair_rows_swapped,
        'lag': 1,
        'suffix': 'lagpanel_d_t_vs_future_a',
        'title_note': ' (swapped axes, second variable at t+1)',
        'ad_pair': ('d', 'a'),
    },
]
for variant in lag_variants_quad:
    _render_quadruple_lagged(**variant)


1+1